# 107 — DepMap RNAi Acquisition and Audit

## Objective

Audit and freeze the RNA-interference dependency resource required for
notebook 501 — RNAi Associations.

The primary RNAi dependency matrix is the combined DEMETER2 resource integrating
Achilles, DRIVE, and Marcotte screens. This historical RNAi resource is treated
as a distinct functional-genomics platform from the DepMap Public 24Q4 CRISPR
gene-effect data used in notebook 500.

## Source files

The acquired resource contains four files selected for the Phase 5 RNAi
handoff:

- `D2_combined_gene_dep_scores.csv`
- `D2_combined_CL_data.csv`
- `sample_info.csv`
- `README.txt`

`D2_combined_gene_dep_scores.csv` is the primary dependency-score matrix.
The remaining files are retained for provenance, sample annotation, and
deterministic cell-line identifier harmonization.

## Audit scope

This notebook will:

1. establish byte-level identity and provenance for the acquired files;
2. characterize the structure and analytical units of the combined DEMETER2
   gene-dependency matrix;
3. determine the cell-line identifier system used by the RNAi resource;
4. define a deterministic mapping to the frozen DepMap model universe without
   fuzzy or result-driven identifier rescue;
5. characterize RNAi coverage of the frozen 713-model Phase 3 cell-line cohort;
6. freeze an analysis-ready model-mapping and coverage handoff for notebook 501.

## Analytical boundary

This notebook is an acquisition and infrastructure audit, not a
functional-vulnerability analysis.

It will not:

- test consensus-program–RNAi associations;
- define gene-level RNAi eligibility thresholds for association testing;
- inspect genes for biological prioritization;
- alter the frozen 713-model Phase 3 cohort;
- alter the three frozen Phase 4 consensus programs;
- use CRISPR results from notebook 500 to select RNAi genes or models.

Any unresolved identifier mappings will remain unresolved rather than being
rescued through fuzzy matching or ad hoc manual decisions.

Notebook 501 will consume only the frozen outputs of this audit and will define
its own prespecified RNAi association universe before inspecting association
results.

In [1]:
# =============================================================================
# Imports
# =============================================================================

import pandas as pd

from pancancer_epigenetics.utils.file_checks import calculate_sha256
from pancancer_epigenetics.utils.paths import Paths, project_relative_path

In [2]:
# =============================================================================
# RNAi input paths
# =============================================================================

RNAI_GENE_DEPENDENCY_PATH = (
    Paths.depmap
    / "D2_combined_gene_dep_scores.csv"
)

RNAI_CELL_LINE_DATA_PATH = (
    Paths.depmap
    / "D2_combined_CL_data.csv"
)

RNAI_SAMPLE_INFO_PATH = (
    Paths.depmap
    / "sample_info.csv"
)

RNAI_README_PATH = (
    Paths.depmap
    / "README.txt"
)

In [7]:
# =============================================================================
# Inventory and identify acquired RNAi files
# =============================================================================

rnai_input_paths = {
    "gene_dependency": RNAI_GENE_DEPENDENCY_PATH,
    "cell_line_data": RNAI_CELL_LINE_DATA_PATH,
    "sample_info": RNAI_SAMPLE_INFO_PATH,
    "readme": RNAI_README_PATH,
}

rnai_file_inventory = pd.DataFrame(
    [
        {
            "resource": resource,
            "relative_path": project_relative_path(path),
            "exists": path.exists(),
            "size_bytes": path.stat().st_size if path.exists() else None,
            "sha256": calculate_sha256(path) if path.exists() else None,
        }
        for resource, path in rnai_input_paths.items()
    ]
)

rnai_file_inventory

,resource,relative_path,exists,size_bytes,sha256
0,gene_dependency,data/raw/depmap/D2_combined_gene_dep_scores.csv,True,160616970,6bfe7b24ef191debccd502b7dfe61c4b0763caba3e02c6...
1,cell_line_data,data/raw/depmap/D2_combined_CL_data.csv,True,68023,d515cbc05de2a3dc6cbdf352d44a97bb7618fc1ffb63e4...
2,sample_info,data/raw/depmap/sample_info.csv,True,76352,8dcbd6da1e4858e7fa5b3910e8cf3feb1045a0c31a2e5d...
3,readme,data/raw/depmap/README.txt,True,9476,d0f6a52e68faedfa4190333317af5e82928ddce41a8df6...


In [10]:
# =============================================================================
# Load RNAi cell-line annotation tables
# =============================================================================

rnai_sample_info = pd.read_csv(RNAI_SAMPLE_INFO_PATH)
rnai_cell_line_data = pd.read_csv(RNAI_CELL_LINE_DATA_PATH)

print("sample_info:", rnai_sample_info.shape)
print("sample_info columns:", rnai_sample_info.columns.tolist())
print()
print("D2_combined_CL_data:", rnai_cell_line_data.shape)
print("D2_combined_CL_data columns:", rnai_cell_line_data.columns.tolist())

sample_info: (712, 14)
sample_info columns: ['CCLE_ID', 'disease', 'disease_subtype', 'disease_sub_subtype', 'in_DRIVE', 'in_Achilles', 'in_Marcotte', 'Novartis_name', 'Novartis_Primary_site', 'Novartis_Pathologist_Annotation', 'Marcotte_name', 'Marcotte_subtype_three_receptor', 'Marcotte_subtype_neve', 'Marcotte_subtype_intrinsic']

D2_combined_CL_data: (712, 6)
D2_combined_CL_data columns: ['Unnamed: 0', 'gene_slope', 'CL_slope', 'noise_vars', 'offset_mean', 'offset_sd']


In [11]:
# =============================================================================
# Compare RNAi cell-line identifiers across annotation tables
# =============================================================================

sample_info_ids = rnai_sample_info["CCLE_ID"].astype("string")
cell_line_data_ids = rnai_cell_line_data["Unnamed: 0"].astype("string")

print(f"sample_info IDs:            {sample_info_ids.nunique():,}")
print(f"D2_combined_CL_data IDs:    {cell_line_data_ids.nunique():,}")
print(
    "Shared IDs:                 "
    f"{len(set(sample_info_ids) & set(cell_line_data_ids)):,}"
)
print(
    "Same IDs in same order:     "
    f"{sample_info_ids.equals(cell_line_data_ids)}"
)

sample_info IDs:            712
D2_combined_CL_data IDs:    712
Shared IDs:                 712
Same IDs in same order:     True


In [12]:
# =============================================================================
# Inspect RNAi dependency-matrix structure
# =============================================================================

rnai_dependency_preview = pd.read_csv(
    RNAI_GENE_DEPENDENCY_PATH,
    nrows=5,
)

print("Preview shape:", rnai_dependency_preview.shape)
print("First columns:", rnai_dependency_preview.columns[:6].tolist())
print("Last columns:", rnai_dependency_preview.columns[-5:].tolist())

rnai_dependency_preview.iloc[:, :6]

Preview shape: (5, 713)
First columns: ['Unnamed: 0', '127399_SOFT_TISSUE', '1321N1_CENTRAL_NERVOUS_SYSTEM', '143B_BONE', '184A1_BREAST', '184B5_BREAST']
Last columns: ['YKG1_CENTRAL_NERVOUS_SYSTEM', 'YMB1_BREAST', 'ZR751_BREAST', 'ZR7530_BREAST', 'ZR75B_BREAST']


,Unnamed: 0,127399_SOFT_TISSUE,1321N1_CENTRAL_NERVOUS_SYSTEM,143B_BONE,184A1_BREAST,184B5_BREAST
0,A1BG (1),NaN,NaN,0.146042,-0.190388,0.907063
1,NAT2 (10),NaN,NaN,0.102854,0.384106,0.403192
2,ADA (100),NaN,NaN,0.168839,-0.120700,0.004394
3,CDH2 (1000),-0.194962,-0.028171,0.063047,-0.237251,-0.017059
4,AKT3 (10000),-0.256108,0.100751,-0.008077,0.060267,-0.094749


In [15]:
# =============================================================================
# Compare dependency-matrix cell-line identifiers
# =============================================================================

dependency_cell_line_ids = pd.Index(
    rnai_dependency_preview.columns[1:],
    dtype="string",
)

print(f"Dependency-matrix IDs:      {dependency_cell_line_ids.nunique():,}")
print(
    "Shared with sample_info:     "
    f"{len(set(dependency_cell_line_ids) & set(sample_info_ids)):,}"
)
print(
    "Same IDs in same order:      "
    f"{dependency_cell_line_ids.equals(pd.Index(sample_info_ids))}"
)

Dependency-matrix IDs:      712
Shared with sample_info:     712
Same IDs in same order:      True


In [16]:
# =============================================================================
# Load combined DEMETER2 gene-dependency matrix
# =============================================================================

rnai_dependency = pd.read_csv(
    RNAI_GENE_DEPENDENCY_PATH,
    index_col=0,
)

rnai_dependency.index.name = "gene"

print(f"RNAi dependency matrix: {rnai_dependency.shape[0]:,} genes × "
      f"{rnai_dependency.shape[1]:,} cell lines")

RNAi dependency matrix: 17,309 genes × 712 cell lines


In [17]:
# =============================================================================
# Parse DEMETER2 gene identifiers
# =============================================================================

rnai_gene_metadata = (
    rnai_dependency.index
    .to_series(index=rnai_dependency.index, name="gene_label")
    .str.extract(
        r"^(?P<gene_symbol>.+) \((?P<entrez_id>\d+)\)$"
    )
    .reset_index(names="gene_label")
)

rnai_gene_metadata["entrez_id"] = pd.to_numeric(
    rnai_gene_metadata["entrez_id"],
    errors="coerce",
).astype("Int64")

print(f"Gene labels:             {len(rnai_gene_metadata):,}")
print(f"Unique gene labels:      {rnai_gene_metadata['gene_label'].nunique():,}")
print(f"Parsed gene identifiers: {rnai_gene_metadata['entrez_id'].notna().sum():,}")
print(f"Unique gene symbols:     {rnai_gene_metadata['gene_symbol'].nunique():,}")
print(f"Unique Entrez IDs:       {rnai_gene_metadata['entrez_id'].nunique():,}")

Gene labels:             17,309
Unique gene labels:      17,309
Parsed gene identifiers: 17,023
Unique gene symbols:     17,023
Unique Entrez IDs:       17,023


In [18]:
# =============================================================================
# Inspect non-standard DEMETER2 gene labels
# =============================================================================

unparsed_gene_labels = rnai_gene_metadata.loc[
    rnai_gene_metadata["entrez_id"].isna(),
    "gene_label",
]

print(f"Non-standard gene labels: {len(unparsed_gene_labels):,}")

unparsed_gene_labels.head(40).to_frame()

Non-standard gene labels: 286


,gene_label
12,GUSBP9&LOC100653061&SMA5&GUSBP2&GUSBP3 (100049...
19,GTF2IP4&GTF2IP1 (100093631&2970)
24,LOC100101478&H2BFXP (100101478&767811)
26,FAM86JP&FAM86C1&FAM86FP (100125556&55199&653113)
33,SLC39A12-AS1&LOC389834 (100129213&389834)
34,LINC01160&LOC105379331&LOC107984772&FAM71F2 (1...
42,HSFX2&HSFX1 (100130086&100506164)
52,RPL21P28&RPL21 (100131205&6144)
56,ZNF705E&ZNF705B (100131539&100132396)
57,LOC100131626&LOC105376489&LOC105379724&LOC1079...


In [20]:
# =============================================================================
# Characterize composite DEMETER2 gene labels
# =============================================================================

rnai_gene_label_parts = (
    rnai_dependency.index
    .to_series(index=rnai_dependency.index, name="gene_label")
    .str.extract(
        r"^(?P<gene_symbol_group>.+) \((?P<entrez_id_group>[\d&]+)\)$"
    )
)

rnai_gene_label_parts["n_symbols"] = (
    rnai_gene_label_parts["gene_symbol_group"]
    .str.count("&")
    .add(1)
)

rnai_gene_label_parts["n_entrez_ids"] = (
    rnai_gene_label_parts["entrez_id_group"]
    .str.count("&")
    .add(1)
)

print(
    "Fully parsed labels:       "
    f"{rnai_gene_label_parts['entrez_id_group'].notna().sum():,}"
)
print(
    "Single-gene labels:        "
    f"{(rnai_gene_label_parts['n_entrez_ids'] == 1).sum():,}"
)
print(
    "Composite gene labels:     "
    f"{(rnai_gene_label_parts['n_entrez_ids'] > 1).sum():,}"
)
print(
    "Symbol/Entrez mismatches:  "
    f"{(rnai_gene_label_parts['n_symbols'] != rnai_gene_label_parts['n_entrez_ids']).sum():,}"
)

Fully parsed labels:       17,309
Single-gene labels:        17,023
Composite gene labels:     286
Symbol/Entrez mismatches:  0


In [21]:
# =============================================================================
# Audit RNAi dependency-matrix completeness
# =============================================================================

total_values = rnai_dependency.size
observed_values = rnai_dependency.notna().sum().sum()
missing_values = total_values - observed_values

print(f"Total dependency values:    {total_values:,}")
print(f"Observed values:            {observed_values:,}")
print(f"Missing values:             {missing_values:,}")
print(f"Missing fraction:           {missing_values / total_values:.4%}")
print(f"Minimum dependency score:   {rnai_dependency.min().min():.6f}")
print(f"Maximum dependency score:   {rnai_dependency.max().max():.6f}")

Total dependency values:    12,324,008
Observed values:            9,551,884
Missing values:             2,772,124
Missing fraction:           22.4937%
Minimum dependency score:   -5.932379
Maximum dependency score:   2.774956


In [22]:
# =============================================================================
# Characterize RNAi coverage by gene and cell line
# =============================================================================

rnai_gene_coverage = rnai_dependency.notna().sum(axis=1)
rnai_cell_line_coverage = rnai_dependency.notna().sum(axis=0)

print("Observed cell lines per gene:")
print(rnai_gene_coverage.describe().round(2))

print()
print("Observed genes per cell line:")
print(rnai_cell_line_coverage.describe().round(2))

Observed cell lines per gene:
count    17309.00
mean       551.84
std        147.55
min         18.00
25%        386.00
50%        547.00
75%        710.00
max        712.00
dtype: float64

Observed genes per cell line:
count      712.00
mean     13415.57
std       3500.92
min       2998.00
25%      11136.00
50%      13380.00
75%      16593.00
max      17309.00
dtype: float64


In [23]:
# =============================================================================
# Load frozen Phase 3 modeling cohort
# =============================================================================

MODELING_COHORT_PATH = (
    Paths.interim
    / "metadata"
    / "302_integrated_modeling_cohort.csv"
)

modeling_cohort = pd.read_csv(MODELING_COHORT_PATH)

print(f"Frozen modeling cohort: {modeling_cohort.shape}")
print("Columns:", modeling_cohort.columns.tolist())

Frozen modeling cohort: (713, 8)
Columns: ['ModelID', 'SangerModelID', 'COSMICID', 'CellLineName', 'OncotreeLineage', 'OncotreePrimaryDisease', 'OncotreeSubtype', 'CCLEName']


In [24]:
# =============================================================================
# Compare historical RNAi CCLE IDs with frozen CCLE names
# =============================================================================

frozen_ccle_names = modeling_cohort["CCLEName"].astype("string")

shared_ccle_names = set(sample_info_ids) & set(frozen_ccle_names)

print(f"RNAi CCLE IDs:             {sample_info_ids.nunique():,}")
print(f"Frozen CCLE names:         {frozen_ccle_names.nunique():,}")
print(f"Exact shared identifiers:  {len(shared_ccle_names):,}")
print(
    f"RNAi IDs without match:    "
    f"{len(set(sample_info_ids) - set(frozen_ccle_names)):,}"
)
print(
    f"Frozen IDs without RNAi:   "
    f"{len(set(frozen_ccle_names) - set(sample_info_ids)):,}"
)

RNAi CCLE IDs:             712
Frozen CCLE names:         713
Exact shared identifiers:  443
RNAi IDs without match:    269
Frozen IDs without RNAi:   270


In [25]:
# =============================================================================
# Inspect current DepMap model identifiers
# =============================================================================

DEPMAP_MODEL_PATH = (
    Paths.depmap
    / "Model.csv"
)

depmap_models = pd.read_csv(DEPMAP_MODEL_PATH)

identifier_columns = [
    column
    for column in depmap_models.columns
    if any(
        token in column.lower()
        for token in ("model", "ccle", "name", "alias")
    )
]

print(f"DepMap model metadata: {depmap_models.shape}")
print("Identifier-related columns:")
print(identifier_columns)


DepMap model metadata: (2105, 47)
Identifier-related columns:
['ModelID', 'CellLineName', 'StrippedCellLineName', 'DepmapModelType', 'ModelType', 'ModelDerivationMaterial', 'ModelTreatment', 'EngineeredModel', 'EngineeredModelDetails', 'CCLEName', 'ModelAvailableInDbgap', 'ModelSubtypeFeatures', 'SangerModelID']


In [26]:
# =============================================================================
# Audit current DepMap CCLE-name structure
# =============================================================================

depmap_name_pairs = (
    depmap_models[
        ["ModelID", "CCLEName", "StrippedCellLineName"]
    ]
    .dropna(subset=["CCLEName", "StrippedCellLineName"])
    .copy()
)

depmap_name_pairs["ccle_starts_with_stripped"] = [
    ccle_name.startswith(f"{stripped_name}_")
    for ccle_name, stripped_name in zip(
        depmap_name_pairs["CCLEName"],
        depmap_name_pairs["StrippedCellLineName"],
    )
]

print(f"Models with both identifiers: {len(depmap_name_pairs):,}")
print(
    "CCLEName consistent with StrippedCellLineName prefix: "
    f"{depmap_name_pairs['ccle_starts_with_stripped'].sum():,}"
)
print(
    "Distinct stripped names: "
    f"{depmap_name_pairs['StrippedCellLineName'].nunique():,}"
)

Models with both identifiers: 1,999
CCLEName consistent with StrippedCellLineName prefix: 1,943
Distinct stripped names: 1,998


In [27]:
# =============================================================================
# Validate historical CCLE-name stripping on exact matches
# =============================================================================

exact_match_validation = (
    rnai_sample_info[["CCLE_ID"]]
    .merge(
        depmap_models[
            ["ModelID", "CCLEName", "StrippedCellLineName"]
        ],
        left_on="CCLE_ID",
        right_on="CCLEName",
        how="inner",
    )
)

exact_match_validation["historical_stripped_name"] = (
    exact_match_validation["CCLE_ID"]
    .str.split("_", n=1)
    .str[0]
)

exact_match_validation["stripped_name_matches"] = (
    exact_match_validation["historical_stripped_name"]
    == exact_match_validation["StrippedCellLineName"]
)

print(f"Exact RNAi–DepMap matches:       {len(exact_match_validation):,}")
print(
    "Historical stripping matches:  "
    f"{exact_match_validation['stripped_name_matches'].sum():,}"
)
print(
    "Historical stripping differs:  "
    f"{(~exact_match_validation['stripped_name_matches']).sum():,}"
)

Exact RNAi–DepMap matches:       701
Historical stripping matches:  698
Historical stripping differs:  3


In [28]:
# =============================================================================
# Inspect historical stripping exceptions
# =============================================================================

stripping_exceptions = exact_match_validation.loc[
    ~exact_match_validation["stripped_name_matches"],
    [
        "CCLE_ID",
        "ModelID",
        "StrippedCellLineName",
        "historical_stripped_name",
    ],
]

stripping_exceptions

,CCLE_ID,ModelID,StrippedCellLineName,historical_stripped_name
108,D341MED_CENTRAL_NERVOUS_SYSTEM,ACH-000095,D341Med,D341MED
643,SYO1_SOFT_TISSUE,ACH-001275,OSA1777,SYO1
666,TT_OESOPHAGUS,ACH-000561,TDOTT,TT


In [30]:
# =============================================================================
# Evaluate stripped-name mapping for unmatched RNAi cell lines
# =============================================================================

unmatched_rnai_ids = sorted(
    set(sample_info_ids) - set(depmap_models["CCLEName"].dropna())
)

unmatched_rnai = pd.DataFrame(
    {"CCLE_ID": unmatched_rnai_ids}
)

unmatched_rnai["historical_stripped_name"] = (
    unmatched_rnai["CCLE_ID"]
    .str.split("_", n=1)
    .str[0]
)

stripped_mapping_candidates = unmatched_rnai.merge(
    depmap_models[
        ["ModelID", "CCLEName", "StrippedCellLineName"]
    ],
    left_on="historical_stripped_name",
    right_on="StrippedCellLineName",
    how="left",
)

candidate_counts = (
    stripped_mapping_candidates
    .groupby("CCLE_ID")["ModelID"]
    .count()
)

print(f"RNAi IDs without exact current match: {len(unmatched_rnai):,}")
print(f"Unique stripped-name matches:         {(candidate_counts == 1).sum():,}")
print(f"No stripped-name match:               {(candidate_counts == 0).sum():,}")
print(f"Ambiguous stripped-name matches:      {(candidate_counts > 1).sum():,}")

RNAi IDs without exact current match: 11
Unique stripped-name matches:         3
No stripped-name match:               8
Ambiguous stripped-name matches:      0


In [31]:
# =============================================================================
# Inspect unmatched historical RNAi identifiers
# =============================================================================

unmatched_mapping_review = (
    stripped_mapping_candidates[
        [
            "CCLE_ID",
            "historical_stripped_name",
            "ModelID",
            "CCLEName",
            "StrippedCellLineName",
        ]
    ]
    .sort_values("CCLE_ID")
    .reset_index(drop=True)
)

unmatched_mapping_review

,CCLE_ID,historical_stripped_name,ModelID,CCLEName,StrippedCellLineName
0,184A1_BREAST,184A1,NaN,NaN,NaN
1,184B5_BREAST,184B5,NaN,NaN,NaN
2,AZ521_STOMACH,AZ521,ACH-001015,AZ521_SMALL_INTESTINE,AZ521
3,COLO699_LUNG,COLO699,NaN,NaN,NaN
4,GISTT1_GASTROINTESTINAL_TRACT,GISTT1,ACH-002332,GISTT1_STOMACH,GISTT1
5,KP1NL_PANCREAS,KP1NL,NaN,NaN,NaN
6,LY2_BREAST,LY2,NaN,NaN,NaN
7,MB157_BREAST,MB157,NaN,NaN,NaN
8,NCIH1339_LUNG,NCIH1339,NaN,NaN,NaN
9,SKRC20_KIDNEY,SKRC20,NaN,NaN,NaN


In [33]:
# =============================================================================
# Build deterministic RNAi-to-DepMap model mapping
# =============================================================================

rnai_model_mapping = (
    rnai_sample_info[["CCLE_ID"]]
    .merge(
        depmap_models[
            ["ModelID", "CCLEName", "StrippedCellLineName"]
        ],
        left_on="CCLE_ID",
        right_on="CCLEName",
        how="left",
    )
)

rnai_model_mapping["mapping_method"] = (
    rnai_model_mapping["ModelID"]
    .notna()
    .map({True: "exact_ccle_name", False: "unresolved"})
)

stripped_lookup = (
    depmap_models[
        ["ModelID", "StrippedCellLineName"]
    ]
    .dropna()
    .drop_duplicates(subset="StrippedCellLineName", keep=False)
    .set_index("StrippedCellLineName")["ModelID"]
)

unresolved_mask = rnai_model_mapping["ModelID"].isna()

historical_stripped = (
    rnai_model_mapping.loc[unresolved_mask, "CCLE_ID"]
    .str.split("_", n=1)
    .str[0]
)

fallback_model_ids = historical_stripped.map(stripped_lookup)

resolved_by_stripped = fallback_model_ids.notna()

rnai_model_mapping.loc[
    fallback_model_ids.index[resolved_by_stripped],
    "ModelID",
] = fallback_model_ids[resolved_by_stripped]

rnai_model_mapping.loc[
    fallback_model_ids.index[resolved_by_stripped],
    "mapping_method",
] = "unique_stripped_name"

print(rnai_model_mapping["mapping_method"].value_counts())

mapping_method
exact_ccle_name         701
unresolved                8
unique_stripped_name      3
Name: count, dtype: int64


In [34]:
# =============================================================================
# Check uniqueness of resolved RNAi model mappings
# =============================================================================

resolved_rnai_mapping = rnai_model_mapping.dropna(
    subset=["ModelID"]
)

model_mapping_counts = (
    resolved_rnai_mapping["ModelID"]
    .value_counts()
)

print(f"Resolved RNAi cell lines:    {len(resolved_rnai_mapping):,}")
print(f"Unique mapped ModelIDs:      {resolved_rnai_mapping['ModelID'].nunique():,}")
print(f"Duplicated mapped ModelIDs:  {(model_mapping_counts > 1).sum():,}")

Resolved RNAi cell lines:    704
Unique mapped ModelIDs:      704
Duplicated mapped ModelIDs:  0


In [35]:
# =============================================================================
# Quantify RNAi coverage of the frozen modeling cohort
# =============================================================================

frozen_model_ids = set(
    modeling_cohort["ModelID"].astype("string")
)

rnai_mapped_model_ids = set(
    resolved_rnai_mapping["ModelID"].astype("string")
)

shared_frozen_models = (
    frozen_model_ids
    & rnai_mapped_model_ids
)

print(f"Frozen modeling cohort:     {len(frozen_model_ids):,}")
print(f"Mapped RNAi models:         {len(rnai_mapped_model_ids):,}")
print(f"Shared frozen models:       {len(shared_frozen_models):,}")
print(
    f"Frozen models without RNAi: "
    f"{len(frozen_model_ids - rnai_mapped_model_ids):,}"
)
print(
    f"RNAi models outside cohort: "
    f"{len(rnai_mapped_model_ids - frozen_model_ids):,}"
)

Frozen modeling cohort:     713
Mapped RNAi models:         704
Shared frozen models:       443
Frozen models without RNAi: 270
RNAi models outside cohort: 261


In [36]:
# =============================================================================
# Build frozen-cohort RNAi model handoff
# =============================================================================

rnai_frozen_model_cohort = (
    resolved_rnai_mapping[
        ["CCLE_ID", "ModelID", "mapping_method"]
    ]
    .merge(
        modeling_cohort,
        on="ModelID",
        how="inner",
    )
)

print(f"RNAi models in frozen cohort: {len(rnai_frozen_model_cohort):,}")

rnai_frozen_model_cohort.head()

RNAi models in frozen cohort: 443


,CCLE_ID,ModelID,mapping_method,SangerModelID,COSMICID,CellLineName,OncotreeLineage,OncotreePrimaryDisease,OncotreeSubtype,CCLEName
0,22RV1_PROSTATE,ACH-000956,exact_ccle_name,SIDM00499,924100.0,22Rv1,Prostate,Prostate Adenocarcinoma,Prostate Adenocarcinoma,22RV1_PROSTATE
1,2313287_STOMACH,ACH-000948,exact_ccle_name,SIDM00980,910924.0,23132/87,Esophagus/Stomach,Esophagogastric Adenocarcinoma,Stomach Adenocarcinoma,2313287_STOMACH
2,697_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,ACH-000070,exact_ccle_name,SIDM01076,906800.0,697,Lymphoid,B-Lymphoblastic Leukemia/Lymphoma,B-Lymphoblastic Leukemia/Lymphoma with t(1;19)...,697_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE
3,769P_KIDNEY,ACH-000411,exact_ccle_name,SIDM00803,910922.0,769-P,Kidney,Renal Cell Carcinoma,Renal Clear Cell Carcinoma,769P_KIDNEY
4,786O_KIDNEY,ACH-000649,exact_ccle_name,SIDM00125,905947.0,786-O,Kidney,Renal Cell Carcinoma,Renal Clear Cell Carcinoma,786O_KIDNEY


In [38]:
# =============================================================================
# Characterize RNAi coverage by frozen lineage
# =============================================================================

rnai_lineage_coverage = (
    modeling_cohort["OncotreeLineage"]
    .value_counts()
    .rename("frozen_models")
    .to_frame()
    .join(
        rnai_frozen_model_cohort["OncotreeLineage"]
        .value_counts()
        .rename("rnai_models"),
        how="left",
    )
    .fillna({"rnai_models": 0})
    .astype({"rnai_models": "int64"})
)

rnai_lineage_coverage["coverage_fraction"] = (
    rnai_lineage_coverage["rnai_models"]
    / rnai_lineage_coverage["frozen_models"]
)

rnai_lineage_coverage

,frozen_models,rnai_models,coverage_fraction
OncotreeLineage,,,
Lung,136,97,0.713235
Lymphoid,86,24,0.279070
Esophagus/Stomach,51,36,0.705882
Breast,47,45,0.957447
Bowel,43,32,0.744186
Skin,37,30,0.810811
CNS/Brain,37,26,0.702703
Ovary/Fallopian Tube,34,24,0.705882
Myeloid,32,15,0.468750


In [40]:
# =============================================================================
# Characterize RNAi screen-source composition
# =============================================================================

rnai_frozen_source_cohort = (
    rnai_frozen_model_cohort
    .merge(
        rnai_sample_info[
            ["CCLE_ID", "in_Achilles", "in_DRIVE", "in_Marcotte"]
        ],
        on="CCLE_ID",
        how="left",
    )
)

rnai_source_composition = (
    rnai_frozen_source_cohort[
        ["in_Achilles", "in_DRIVE", "in_Marcotte"]
    ]
    .value_counts()
    .rename("n_models")
    .reset_index()
)

rnai_source_composition


,in_Achilles,in_DRIVE,in_Marcotte,n_models
0,True,True,False,165
1,True,False,False,144
2,False,True,False,93
3,True,True,True,16
4,True,False,True,12
5,False,False,True,8
6,False,True,True,5


In [42]:
# =============================================================================
# Characterize RNAi screen-source composition by lineage
# =============================================================================

rnai_frozen_source_cohort["source_pattern"] = (
    rnai_frozen_source_cohort[
        ["in_Achilles", "in_DRIVE", "in_Marcotte"]
    ]
    .apply(
        lambda row: "+".join(
            source
            for source, included in zip(
                ["Achilles", "DRIVE", "Marcotte"],
                row,
            )
            if included
        ),
        axis=1,
    )
)

rnai_source_by_lineage = pd.crosstab(
    rnai_frozen_source_cohort["OncotreeLineage"],
    rnai_frozen_source_cohort["source_pattern"],
)

rnai_source_by_lineage

source_pattern,Achilles,Achilles+DRIVE,Achilles+DRIVE+Marcotte,Achilles+Marcotte,DRIVE,DRIVE+Marcotte,Marcotte
OncotreeLineage,,,,,,,
Bladder/Urinary Tract,0,3,0,0,6,0,0
Bone,2,6,0,0,0,0,0
Bowel,4,15,0,0,13,0,0
Breast,4,0,16,12,0,5,8
CNS/Brain,9,10,0,0,7,0,0
Cervix,2,0,0,0,0,0,0
Esophagus/Stomach,12,20,0,0,4,0,0
Head and Neck,4,4,0,0,3,0,0
Kidney,0,11,0,0,4,0,0


In [44]:
# =============================================================================
# Characterize gene coverage in the shared frozen RNAi cohort
# =============================================================================

shared_rnai_dependency = rnai_dependency[
    rnai_frozen_model_cohort["CCLE_ID"]
]

shared_rnai_gene_coverage = (
    shared_rnai_dependency
    .notna()
    .sum(axis=1)
)

print(f"Shared RNAi cohort: {shared_rnai_dependency.shape[1]:,} models")
print()
print("Observed shared models per gene:")
print(shared_rnai_gene_coverage.describe().round(2))

Shared RNAi cohort: 443 models

Observed shared models per gene:
count    17309.00
mean       349.32
std         90.58
min         16.00
25%        272.00
50%        350.00
75%        443.00
max        443.00
dtype: float64


In [45]:
# =============================================================================
# Build shared-cohort RNAi gene coverage table
# =============================================================================

rnai_gene_coverage_table = (
    rnai_gene_label_parts
    .rename_axis("gene_label")
    .reset_index()
)

rnai_gene_coverage_table["is_composite"] = (
    rnai_gene_coverage_table["n_entrez_ids"] > 1
)

rnai_gene_coverage_table["observed_models"] = (
    rnai_gene_coverage_table["gene_label"]
    .map(shared_rnai_gene_coverage)
)

rnai_gene_coverage_table["coverage_fraction"] = (
    rnai_gene_coverage_table["observed_models"]
    / shared_rnai_dependency.shape[1]
)

rnai_gene_coverage_table.head()

,gene_label,gene_symbol_group,entrez_id_group,n_symbols,n_entrez_ids,is_composite,observed_models,coverage_fraction
0,A1BG (1),A1BG,1,1,1,False,350,0.790068
1,NAT2 (10),NAT2,10,1,1,False,350,0.790068
2,ADA (100),ADA,100,1,1,False,350,0.790068
3,CDH2 (1000),CDH2,1000,1,1,False,443,1.000000
4,AKT3 (10000),AKT3,10000,1,1,False,443,1.000000


In [47]:
# =============================================================================
# Build shared-cohort RNAi model coverage table
# =============================================================================

shared_rnai_model_coverage = (
    shared_rnai_dependency
    .notna()
    .sum(axis=0)
    .rename("observed_genes")
)

rnai_model_coverage_table = (
    rnai_frozen_source_cohort
    .assign(
        observed_genes=lambda df: (
            df["CCLE_ID"].map(shared_rnai_model_coverage)
        )
    )
)

rnai_model_coverage_table["coverage_fraction"] = (
    rnai_model_coverage_table["observed_genes"]
    / shared_rnai_dependency.shape[0]
)

rnai_model_coverage_table[
    [
        "CCLE_ID",
        "ModelID",
        "OncotreeLineage",
        "source_pattern",
        "observed_genes",
        "coverage_fraction",
    ]
].head()

,CCLE_ID,ModelID,OncotreeLineage,source_pattern,observed_genes,coverage_fraction
0,22RV1_PROSTATE,ACH-000956,Prostate,Achilles,11136,0.643365
1,2313287_STOMACH,ACH-000948,Esophagus/Stomach,Achilles+DRIVE,17237,0.995840
2,697_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,ACH-000070,Lymphoid,Achilles+DRIVE,13380,0.773008
3,769P_KIDNEY,ACH-000411,Kidney,Achilles+DRIVE,17238,0.995898
4,786O_KIDNEY,ACH-000649,Kidney,Achilles+DRIVE,13380,0.773008


In [48]:
print(rnai_model_coverage_table[
    [
        "CCLE_ID",
        "ModelID",
        "OncotreeLineage",
        "source_pattern",
        "observed_genes",
        "coverage_fraction",
    ]
].head())

                                  CCLE_ID     ModelID    OncotreeLineage  \
0                          22RV1_PROSTATE  ACH-000956           Prostate   
1                         2313287_STOMACH  ACH-000948  Esophagus/Stomach   
2  697_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE  ACH-000070           Lymphoid   
3                             769P_KIDNEY  ACH-000411             Kidney   
4                             786O_KIDNEY  ACH-000649             Kidney   

   source_pattern  observed_genes  coverage_fraction  
0        Achilles           11136           0.643365  
1  Achilles+DRIVE           17237           0.995840  
2  Achilles+DRIVE           13380           0.773008  
3  Achilles+DRIVE           17238           0.995898  
4  Achilles+DRIVE           13380           0.773008  


In [49]:
# =============================================================================
# Summarize RNAi model coverage by screen source
# =============================================================================

rnai_coverage_by_source = (
    rnai_model_coverage_table
    .groupby("source_pattern")["observed_genes"]
    .agg(
        n_models="size",
        min_genes="min",
        median_genes="median",
        max_genes="max",
    )
    .sort_values("n_models", ascending=False)
)

rnai_coverage_by_source

,n_models,min_genes,median_genes,max_genes
source_pattern,,,,
Achilles+DRIVE,165,11875,17081.0,17238
Achilles,144,11091,16593.0,16593
DRIVE,93,6177,8385.0,8387
Achilles+DRIVE+Marcotte,16,16250,17308.5,17309
Achilles+Marcotte,12,15726,16247.0,16768
Marcotte,8,15256,15256.0,15256
DRIVE+Marcotte,5,16335,16335.0,16335


In [51]:
# =============================================================================
# Build ModelID-indexed RNAi dependency handoff
# =============================================================================

ccle_to_model = (
    rnai_frozen_model_cohort
    .set_index("CCLE_ID")["ModelID"]
)

rnai_dependency_handoff = (
    shared_rnai_dependency
    .rename(columns=ccle_to_model)
)

print(
    f"RNAi dependency handoff: "
    f"{rnai_dependency_handoff.shape[0]:,} genes × "
    f"{rnai_dependency_handoff.shape[1]:,} frozen models"
)
print(
    f"Unique ModelID columns: "
    f"{rnai_dependency_handoff.columns.nunique():,}"
)

RNAi dependency handoff: 17,309 genes × 443 frozen models
Unique ModelID columns: 443


In [52]:
# =============================================================================
# Persist RNAi audit and handoff artifacts
# =============================================================================

RNAI_INTERIM_DIR = Paths.interim / "dependencies"
RNAI_INTERIM_DIR.mkdir(parents=True, exist_ok=True)

RNAI_MODEL_MAPPING_PATH = (
    RNAI_INTERIM_DIR / "107_rnai_model_mapping.csv"
)
RNAI_FROZEN_COHORT_PATH = (
    RNAI_INTERIM_DIR / "107_rnai_frozen_model_cohort.csv"
)
RNAI_GENE_COVERAGE_PATH = (
    RNAI_INTERIM_DIR / "107_rnai_gene_coverage.csv"
)
RNAI_MODEL_COVERAGE_PATH = (
    RNAI_INTERIM_DIR / "107_rnai_model_coverage.csv"
)
RNAI_DEPENDENCY_HANDOFF_PATH = (
    RNAI_INTERIM_DIR / "107_rnai_dependency_handoff.parquet"
)

rnai_dependency_handoff.index.name = "gene_label"

rnai_model_mapping.to_csv(
    RNAI_MODEL_MAPPING_PATH,
    index=False,
)
rnai_frozen_model_cohort.to_csv(
    RNAI_FROZEN_COHORT_PATH,
    index=False,
)
rnai_gene_coverage_table.to_csv(
    RNAI_GENE_COVERAGE_PATH,
    index=False,
)
rnai_model_coverage_table.to_csv(
    RNAI_MODEL_COVERAGE_PATH,
    index=False,
)
rnai_dependency_handoff.to_parquet(
    RNAI_DEPENDENCY_HANDOFF_PATH,
)

print("RNAi handoff artifacts written: 5")
print(f"Directory: {project_relative_path(RNAI_INTERIM_DIR)}")


RNAi handoff artifacts written: 5
Directory: data/interim/dependencies


In [53]:
# =============================================================================
# Persist RNAi raw-input inventory
# =============================================================================

RNAI_INPUT_INVENTORY_PATH = (
    RNAI_INTERIM_DIR / "107_rnai_input_inventory.csv"
)

rnai_file_inventory.to_csv(
    RNAI_INPUT_INVENTORY_PATH,
    index=False,
)

print(
    "RNAi raw-input inventory written: "
    f"{project_relative_path(RNAI_INPUT_INVENTORY_PATH)}"
)

RNAi raw-input inventory written: data/interim/dependencies/107_rnai_input_inventory.csv


In [54]:
# =============================================================================
# Verify published RNAi audit artifacts
# =============================================================================

rnai_audit_artifact_paths = [
    RNAI_INPUT_INVENTORY_PATH,
    RNAI_MODEL_MAPPING_PATH,
    RNAI_FROZEN_COHORT_PATH,
    RNAI_GENE_COVERAGE_PATH,
    RNAI_MODEL_COVERAGE_PATH,
    RNAI_DEPENDENCY_HANDOFF_PATH,
]

published_artifacts = [
    path
    for path in rnai_audit_artifact_paths
    if path.exists() and path.stat().st_size > 0
]

print(
    f"RNAi audit artifacts verified: "
    f"{len(published_artifacts)}/{len(rnai_audit_artifact_paths)}"
)

RNAi audit artifacts verified: 6/6


# 107 — Completion

The DepMap DEMETER2 RNAi resource has been acquired, audited, harmonized, and prepared for downstream Phase 5 analysis.

## Final audited resource

The combined DEMETER2 dependency matrix contains:

- 17,309 DEMETER2 gene-target units;
- 712 historical RNAi cell lines;
- 9,551,884 observed dependency values;
- 22.49% overall missingness.

Gene-target labels were preserved in their original DEMETER2 representation:

- 17,023 single-gene targets;
- 286 composite targets containing multiple gene symbols and Entrez IDs.

Composite targets were not expanded or reassigned during this audit.

## Cell-line harmonization

The RNAi resource uses historical CCLE identifiers.

Mapping to current DepMap `ModelID` was performed deterministically using:

1. exact `CCLEName` matching;
2. unique `StrippedCellLineName` matching only when no exact match was available.

The final mapping contains:

- 701 exact `CCLEName` mappings;
- 3 unique stripped-name mappings;
- 8 unresolved historical RNAi cell lines;
- 704 resolved and unique current DepMap `ModelID` mappings.

No fuzzy matching or ad hoc identifier rescue was used.

## Frozen-cohort overlap

Intersection with the frozen Phase 3 modeling cohort yielded:

- 713 frozen Phase 3 models;
- 704 mapped RNAi models;
- 443 shared frozen models;
- 270 frozen models without RNAi coverage;
- 261 mapped RNAi models outside the frozen analytical cohort.

Only the 443 shared frozen models are retained for downstream RNAi association analysis.

RNAi coverage is lineage-dependent and therefore not representative of the
full frozen cohort uniformly.

## Screen-source structure

The combined DEMETER2 resource integrates Achilles, DRIVE, and Marcotte screens.

Screen-source composition and gene coverage are heterogeneous across models and
lineages. Marcotte contribution is particularly concentrated in breast models,
while DRIVE-only models show substantially lower gene coverage than several
other source combinations.

These source effects are retained as technical context for notebook 501 but
are not used here to redefine the cohort or modify dependency scores.

## Frozen handoff artifacts

The following artifacts were written under `data/interim/dependencies/`:

- `107_rnai_input_inventory.csv`
- `107_rnai_model_mapping.csv`
- `107_rnai_frozen_model_cohort.csv`
- `107_rnai_gene_coverage.csv`
- `107_rnai_model_coverage.csv`
- `107_rnai_dependency_handoff.parquet`

The final dependency handoff contains:

- 17,309 DEMETER2 gene-target units;
- 443 frozen Phase 3 models;
- current DepMap `ModelID` identifiers.

All six audit artifacts were successfully published and verified.

## Downstream boundary

Notebook 501 — RNAi Associations will consume these frozen handoffs directly.

Notebook 501 must define its gene-eligibility rule and association-testing
framework before inspecting RNAi association results. It must not repeat the
identifier harmonization performed here, expand composite DEMETER2 targets,
incorporate RNAi models outside the frozen cohort, or use notebook 500 CRISPR
results to determine RNAi eligibility.

Notebook 107 is complete.